In [1]:
import os
import sys
from pathlib import Path

REPO = Path("/home/ibm-user/shared/trustpause-xiaowei")

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

os.environ.setdefault(
    "GRANITE_MODEL_PATH",
    "/home/ibm-user/granite-4.0-h-350m-base-be.f16.gguf",
)

from src.explain import GraniteExplainer

print("repository:", REPO)
print("model_path:", os.environ["GRANITE_MODEL_PATH"])
print("import: OK")

repository: /home/ibm-user/shared/trustpause-xiaowei
model_path: /home/ibm-user/granite-4.0-h-350m-base-be.f16.gguf
import: OK


In [2]:
explainer = GraniteExplainer(timeout_seconds=15)

payload = {
    "decision": "pause",
    "score": 0.91,
    "amount_band": "high",
    "reasons": [
        {
            "code": "HIGH_FAN_IN",
            "sentence": "The receiver has many recent incoming transfers.",
        },
        {
            "code": "PASS_THROUGH",
            "sentence": "Funds move onward soon after receipt.",
        },
    ],
}

result = explainer.explain(payload)

print("used_granite:", result["used_granite"])
print("prompt_has_identifiers:", result["prompt_has_identifiers"])
print("generation_seconds:", result["generation_seconds"])
print()
print("warning_en:")
print(result["warning_en"])
print()
print("warning_bm:")
print(result["warning_bm"])
print()
print("analyst_note:")
print(result["analyst_note"])

llama_kv_cache: the V embeddings have different sizes across layers and FA is not enabled - padding V cache to 256


used_granite: True
prompt_has_identifiers: False
generation_seconds: {'warning_en': 4.1931, 'warning_bm': 6.0213, 'analyst_note': 10.2826}

warning_en:
Please pause and check the recipient before continuing.

warning_bm:
Sila berhenti seketika dan semak penerima sebelum meneruskan.

analyst_note:
- Pattern: reason codes HIGH_FAN_IN, PASS_THROUGH.
- Evidence: decision pause, score 0.91, amount band high.
- Suggested action: review the transfer and pause it if the evidence is concerning.


In [5]:
cases = [
    {
        "decision": "pause",
        "score": 0.91,
        "amount_band": "high",
        "reasons": [
            {
                "code": "HIGH_FAN_IN",
                "sentence": "Many recent incoming transfers.",
            },
        ],
    },
    {
        "decision": "review",
        "score": 0.68,
        "amount_band": "medium",
        "reasons": [
            {
                "code": "NEW_RECEIVER",
                "sentence": "The receiver was seen recently.",
            },
            {
                "code": "UNUSUAL_AMOUNT",
                "sentence": "The amount differs from normal history.",
            },
        ],
    },
    {
        "decision": "pause",
        "score": 0.88,
        "amount_band": "high",
        "reasons": [
            {
                "code": "PASS_THROUGH",
                "sentence": "Funds move onward shortly after receipt.",
            },
            {
                "code": "HIGH_OUTFLOW",
                "sentence": "The receiver sends funds to many accounts.",
            },
        ],
    },
    {
        "decision": "hold",
        "score": 0.74,
        "amount_band": "low",
        "reasons": [
            {
                "code": "TIME_PATTERN",
                "sentence": "The transfer occurs during an unusual hour.",
            },
        ],
    },
    {
        "decision": "pause",
        "score": 0.97,
        "amount_band": "high",
        "reasons": [
            {
                "code": "NETWORK_RISK",
                "sentence": "The transfer is close to a known risky pattern.",
            },
            {
                "code": "RAPID_MOVEMENT",
                "sentence": "Funds move quickly through the receiver.",
            },
        ],
    },
]

for number, case in enumerate(cases, start=1):
    output = explainer.explain(case)
    note_lines = output["analyst_note"].splitlines()

    assert output["warning_en"]
    assert output["warning_bm"]
    assert output["used_granite"] is True
    assert output["prompt_has_identifiers"] is False
    assert len(note_lines) == 3
    assert note_lines[0].startswith("- Pattern:")
    assert note_lines[1].startswith("- Evidence:")
    assert note_lines[2].startswith("- Suggested action:")

    print(
        f"case_{number}:",
        output["generation_seconds"],
    )
    print("  English:", output["warning_en"])
    print("  BM:", output["warning_bm"])
    print("  Analyst:", " ".join(note_lines))

print("five notebook cases passed")

case_1: {'warning_en': 3.7731, 'warning_bm': 5.5419, 'analyst_note': 7.8814}
  English: Please pause and check the recipient before continuing.
  BM: Sila berhenti seketika dan semak penerima sebelum meneruskan.
  Analyst: - Pattern: reason codes HIGH_FAN_IN. - Evidence: decision pause, score 0.91, amount band high. - Suggested action: review the transfer and pause it if the evidence is concerning.
case_2: {'warning_en': 3.8814, 'warning_bm': 6.0204, 'analyst_note': 10.0659}
  English: Please review the recipient before continuing.
  BM: Sila berhenti seketika dan semak penerima sebelum meneruskan.
  Analyst: - Pattern: reason codes NEW_RECEIVER, UNUSUAL_AMOUNT. - Evidence: decision review, score 0.68, amount band medium. - Suggested action: review the transfer and pause it if the evidence is concerning.
case_3: {'warning_en': 4.1688, 'warning_bm': 6.088, 'analyst_note': 10.7285}
  English: Please pause and check the recipient before continuing.
  BM: Sila berhenti seketika dan semak p

In [6]:
print("=== GRANITE PRIVACY AUDIT PROMPT ===")
print(explainer.last_prompt)
print("=== END PROMPT ===")
print(
    "prompt_has_identifiers:",
    explainer.last_prompt_has_identifiers,
)

assert explainer.last_prompt
assert explainer.last_prompt_has_identifiers is False

print("privacy prompt check passed")

=== GRANITE PRIVACY AUDIT PROMPT ===
You explain an existing TrustPause risk decision. Do not make or change the risk decision. Use only the supplied information. Do not invent personal data.

Write one calm, non-blaming English warning for the sender. Tell them what to check and that they can cancel the transfer. Use no more than 60 words.

Example:
Decision: pause
Risk score: 0.80
Amount band: medium
Reason: unusual transfer pattern
Answer: Please pause and check the recipient before continuing.

Decision: pause
Risk score: 0.970
Amount band: high
Reason codes and explanations: [{"code":"NETWORK_RISK","sentence":"The transfer is close to a known risky pattern."},{"code":"RAPID_MOVEMENT","sentence":"Funds move quickly through the receiver."}]
Answer:

You explain an existing TrustPause risk decision. Do not make or change the risk decision. Use only the supplied information. Do not invent personal data.

Tulis satu amaran ringkas dalam Bahasa Malaysia yang tenang dan tidak menyalahkan